# Week 2: Row Serialization & Prompt Engineering
**Goal:** Convert tabular rows from `personas_sample_500.csv` into natural language prompts,
build zero-shot and few-shot prompt functions, and manually test 10 rows through Nemotron
to verify the pipeline works before running full experiments in Week 3.

By the end of this notebook we will have:
- `src/serialize.py` — finalized and tested
- `src/prompts.py` — zero-shot and few-shot builders
- `src/parse_response.py` — label + reasoning trace extractor
- 10 manual test results with reasoning traces saved to `results/week2_manual_tests.csv`

## 1. Imports

In [1]:
import pandas as pd
import requests
import json
import time
import re
import os

print("Imports OK")

Imports OK


## 2. Loading the 500-row Test Sample

In [2]:
sample = pd.read_csv("../data/personas_sample_500.csv")

sample = sample[
    ~sample["occupation"].str.lower().str.strip().isin([
        "no occupation", "not in workforce", "not_in_workforce", ""
    ])
].reset_index(drop=True)

print(f"Rows after filtering uninformative occupations: {len(sample)}")
print(f"Label balance after filter:")
print(sample["label_name"].value_counts())

print(f"Sample shape: {sample.shape}")
print(f"Label balance:")
print(sample["label_name"].value_counts())
print()
sample.head(3)

s = pd.read_csv("../data/personas_sample_500.csv")
print(f"Total rows: {len(s)}")
print(f"No occupation rows: {s[s['occupation'].str.lower().str.strip() == 'no occupation'].shape[0]}")
print(s["occupation"].value_counts().tail(5))



Rows after filtering uninformative occupations: 500
Label balance after filter:
label_name
college        256
not_college    244
Name: count, dtype: int64
Sample shape: (500, 8)
Label balance:
label_name
college        256
not_college    244
Name: count, dtype: int64

Total rows: 500
No occupation rows: 0
occupation
paramedic                                                1
budget_analyst                                           1
insurance_claims_or_policy_processing_clerk              1
designer                                                 1
chemical_processing_machine_setter_operator_or_tender    1
Name: count, dtype: int64


## 3. Row Serialization

This converts one Nemotron-Personas-USA row into a natural language string that Nemotron can reason about.
The occupation column is the most semantically rich feature. We will keep the full name. 

In [3]:
def serialize_row(row: dict) -> str:
    
    occupation = str(row["occupation"]).replace("_", " ").strip()
    marital    = str(row["marital_status"]).replace("_", " ").strip()
    sex        = str(row["sex"]).lower().strip()
    state      = str(row["state"]).strip()
    age        = int(row["age"])

    return (
        f"A {age}-year-old {sex}, {marital}, "
        f"working as a {occupation}. "
        f"Located in {state}."
    )

# Test on 5 rows
print("Serialization examples")
print()
for _, row in sample.sample(5, random_state=7).iterrows():
    print(f"Input:  {serialize_row(row.to_dict())}")
    print(f"Label:  {row['label_name']} ({row['education_level']})")
    print()

Serialization examples

Input:  A 48-year-old female, married present, working as a software developer. Located in AZ.
Label:  not_college (associates)

Input:  A 66-year-old female, married present, working as a occupational therapist. Located in KS.
Label:  not_college (some_college)

Input:  A 60-year-old male, divorced, working as a financial manager. Located in VA.
Label:  not_college (associates)

Input:  A 38-year-old female, married present, working as a database administrator or architect. Located in UT.
Label:  college (graduate)

Input:  A 34-year-old male, married present, working as a no occupation. Located in NJ.
Label:  not_college (some_college)



## 4. Prompt Builders

In [4]:
SYSTEM_PROMPT = """You are an education level classifier. Given a person's demographic information,
predict whether they are college-educated (have a bachelor's degree or higher) or not.
Think step by step, then respond with ONLY one of these two labels on the final line: college or not_college."""


def build_zero_shot_prompt(row: dict) -> list:
    # Build a zero-shot chat prompt for one row
    description = serialize_row(row)
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": (
            f"Classify this person's education level:\n\n"
            f"{description}\n\n"
            f"Answer with college or not_college only."
        )}
    ]


def build_few_shot_prompt(row: dict, examples: list) -> list:

    # Build a few-shot chat prompt for one row.
    # examples: list of dicts with keys 'row' (dict) and 'label' (str)

    example_text = ""
    for ex in examples:
        example_text += f"Person: {serialize_row(ex['row'])}\nLabel: {ex['label']}\n\n"

    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": (
            f"Here are some labeled examples:\n\n{example_text}"
            f"Now classify this person:\n\n"
            f"{serialize_row(row)}\n\n"
            f"Answer with college or not_college only."
        )}
    ]


# Preview both prompt types on one row
test_row = sample.iloc[0].to_dict()

print("=== ZERO-SHOT PROMPT ===")
zs = build_zero_shot_prompt(test_row)
for msg in zs:
    print(f"[{msg['role'].upper()}]\n{msg['content']}\n")

print("\n=== FEW-SHOT PROMPT (3 examples) ===")
few_shot_pool = sample[sample["label_name"] == "college"].head(2).to_dict("records") + \
                sample[sample["label_name"] == "not_college"].head(1).to_dict("records")
examples = [{"row": r, "label": r["label_name"]} for r in few_shot_pool]
fs = build_few_shot_prompt(test_row, examples)
for msg in fs:
    print(f"[{msg['role'].upper()}]\n{msg['content']}\n")

=== ZERO-SHOT PROMPT ===
[SYSTEM]
You are an education level classifier. Given a person's demographic information,
predict whether they are college-educated (have a bachelor's degree or higher) or not.
Think step by step, then respond with ONLY one of these two labels on the final line: college or not_college.

[USER]
Classify this person's education level:

A 25-year-old female, never married, working as a computer or information research scientist. Located in PA.

Answer with college or not_college only.


=== FEW-SHOT PROMPT (3 examples) ===
[SYSTEM]
You are an education level classifier. Given a person's demographic information,
predict whether they are college-educated (have a bachelor's degree or higher) or not.
Think step by step, then respond with ONLY one of these two labels on the final line: college or not_college.

[USER]
Here are some labeled examples:

Person: A 25-year-old female, never married, working as a computer or information research scientist. Located in PA.
Labe

## 5. Response Parser

Nemotron 3 Nano wraps its reasoning in `<think>...</think>` tags.
We extract the reasoning trace and the final label separately.

In [5]:
def parse_response(content: str) -> tuple:
    """
    Extract label and reasoning trace from Nemotron's response.
    Returns: (label, trace) where label is 'college', 'not_college', or 'UNKNOWN'
    """
    # Extract <think>...</think> reasoning trace
    think_match = re.search(r"<think>(.*?)</think>", content, re.DOTALL)
    trace = think_match.group(1).strip() if think_match else ""

    # Search for label where not_college must come before college in the regex
    # to avoid matching 'college' inside 'not_college'
    label_match = re.search(r"\b(not_college|college)\b", content, re.IGNORECASE)
    label = label_match.group(1).lower() if label_match else "UNKNOWN"

    return label, trace


# Test the parser on mock responses
mock_responses = [
    "<think>This person works as a surgeon which requires medical school.\nThey are highly educated.</think>\ncollege",
    "<think>Fast food worker typically does not require a college degree.</think>\nnot_college",
    "Based on the information provided, I would classify this as: not_college",
    "I cannot determine the answer.",  # UNKNOWN case
]

print("=== Parser tests ===")
for resp in mock_responses:
    label, trace = parse_response(resp)
    print(f"Label: {label:12s} | Trace: {trace[:60]}..." if trace else f"Label: {label:12s} | No trace")

=== Parser tests ===
Label: college      | Trace: This person works as a surgeon which requires medical school...
Label: college      | Trace: Fast food worker typically does not require a college degree...
Label: not_college  | No trace
Label: UNKNOWN      | No trace


## 6. Ollama Setup Check

Before running inference, verify Ollama is installed and the Nemotron model is available.

> **If Ollama is not installed yet:** Download from https://ollama.com and install it.
> Then in a terminal run: `ollama pull nemotron-mini` or `ollama pull hf.co/nvidia/NVIDIA-Nemotron-3-Nano-4B-GGUF`
> Keep Ollama running in the background while using this notebook.

In [6]:
OLLAMA_URL = "http://localhost:11434/v1/chat/completions"
MODEL_NAME = "hf.co/nvidia/NVIDIA-Nemotron-3-Nano-4B-GGUF:latest"  # update this if you use a different model tag

def check_ollama():
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=5)
        models = [m["name"] for m in r.json().get("models", [])]
        print(f"Ollama is running. Available models: {models}")
        return True
    except Exception as e:
        print(f"Ollama not reachable: {e}")
        print("Make sure Ollama is installed and running (ollama serve)")
        return False

ollama_ready = check_ollama()

Ollama is running. Available models: ['hf.co/nvidia/NVIDIA-Nemotron-3-Nano-4B-GGUF:latest', 'nemotron-3-nano:4b', 'nemotron-mini:latest']


## 7. Inference Function

In [7]:
def classify_row(messages: list, enable_thinking: bool = True, timeout: int = 120) -> dict:

    # Send a prompt to Ollama and return label, reasoning trace, timing, and token count.

    payload = {
        "model": MODEL_NAME,
        "messages": messages,
        "max_tokens": 1024,
        "temperature": 0.1,   # low temperature for more consistent labels
        "stream": False,
    }

    # Nemotron thinking toggle  append /no_think to disable reasoning)
    if not enable_thinking:
        payload["messages"] = [
            {"role": m["role"], "content": m["content"] + (" /no_think" if m["role"] == "user" else "")}
            for m in messages
        ]

    start = time.time()
    try:
        response = requests.post(OLLAMA_URL, json=payload, timeout=timeout).json()
        elapsed = time.time() - start
        content = response["choices"][0]["message"]["content"]
        tokens  = response.get("usage", {}).get("completion_tokens", 0)
        label, trace = parse_response(content)
    except Exception as e:
        elapsed = time.time() - start
        content, label, trace, tokens = str(e), "ERROR", "", 0

    return {
        "label":   label,
        "trace":   trace,
        "raw":     content,
        "time_ms": round(elapsed * 1000),
        "tokens":  tokens,
    }

print("classify_row() defined.")

classify_row() defined.


## 8. Manual Test with 10 Rows

Run zero-shot classification on 10 rows and inspect the reasoning traces.
This is Javi's sanity check before running the full 500-row experiment in Week 3.

In [8]:
if not ollama_ready:
    print("Skipping — Ollama not running. Come back to this cell after Ollama is set up.")
else:
    test_10 = sample.sample(10, random_state=7).reset_index(drop=True)
    manual_results = []

    for i, row in test_10.iterrows():
        row_dict  = row.to_dict()
        messages  = build_zero_shot_prompt(row_dict)
        result    = classify_row(messages, enable_thinking=True)
        true_label = row["label_name"]
        correct    = result["label"] == true_label

        print(f"Row {i+1:2d} | Input: {serialize_row(row_dict)}")
        print(f"       | True: {true_label:12s} | Predicted: {result['label']:12s} | {'✓' if correct else '✗'} | {result['time_ms']}ms")
        if result["trace"]:
            print(f"       | Reasoning: {result['trace'][:120]}...")
        print()

        manual_results.append({
            "input":       serialize_row(row_dict),
            "true_label":  true_label,
            "pred_label":  result["label"],
            "correct":     correct,
            "time_ms":     result["time_ms"],
            "tokens":      result["tokens"],
            "trace":       result["trace"],
            "raw":         result["raw"],
        })

    results_df = pd.DataFrame(manual_results)
    accuracy   = results_df["correct"].mean()
    print(f"Manual test accuracy (10 rows): {accuracy:.0%}")

    os.makedirs("../results", exist_ok=True)
    results_df.to_csv("../results/week2_manual_tests.csv", index=False)
    print("Saved: results/week2_manual_tests.csv")

Row  1 | Input: A 48-year-old female, married present, working as a software developer. Located in AZ.
       | True: not_college  | Predicted: college      | ✗ | 4332ms

Row  2 | Input: A 66-year-old female, married present, working as a occupational therapist. Located in KS.
       | True: not_college  | Predicted: not_college  | ✓ | 2461ms

Row  3 | Input: A 60-year-old male, divorced, working as a financial manager. Located in VA.
       | True: not_college  | Predicted: college      | ✗ | 2468ms

Row  4 | Input: A 38-year-old female, married present, working as a database administrator or architect. Located in UT.
       | True: college      | Predicted: college      | ✓ | 2496ms

Row  5 | Input: A 34-year-old male, married present, working as a no occupation. Located in NJ.
       | True: not_college  | Predicted: not_college  | ✓ | 2562ms

Row  6 | Input: A 34-year-old female, never married, working as a cashier. Located in NC.
       | True: not_college  | Predicted: not_colleg

## 9. Inspect Best and Worst Reasoning Traces

Look at which rows the model got right vs wrong and why.

In [9]:
try:
    results_df
except NameError:
    results_df = pd.read_csv("../results/week2_manual_tests.csv")

print("=== CORRECT predictions — reasoning traces ===")
for _, row in results_df[results_df["correct"] == True].head(2).iterrows():
    print(f"Input:     {row['input']}")
    print(f"Label:     {row['true_label']}")
    print(f"Reasoning: {row['trace']}")
    print()

print("\n=== WRONG predictions — reasoning traces ===")
for _, row in results_df[results_df["correct"] == False].head(2).iterrows():
    print(f"Input:     {row['input']}")
    print(f"True:      {row['true_label']} | Predicted: {row['pred_label']}")
    print(f"Reasoning: {row['trace']}")
    print()

=== CORRECT predictions — reasoning traces ===
Input:     A 66-year-old female, married present, working as a occupational therapist. Located in KS.
Label:     not_college
Reasoning: 

Input:     A 38-year-old female, married present, working as a database administrator or architect. Located in UT.
Label:     college
Reasoning: 


=== WRONG predictions — reasoning traces ===
Input:     A 48-year-old female, married present, working as a software developer. Located in AZ.
True:      not_college | Predicted: college
Reasoning: 

Input:     A 60-year-old male, divorced, working as a financial manager. Located in VA.
True:      not_college | Predicted: college
Reasoning: 



## 10. Save src/ Files

Write the finalized functions to the src/ Python files for use in Weeks 3 and 4.

In [10]:
serialize_code = '''
def serialize_row(row: dict) -> str:
    """
    Convert one Nemotron-Personas-USA row into a natural language description.
    """
    occupation = str(row["occupation"]).replace("_", " ").strip()
    marital    = str(row["marital_status"]).replace("_", " ").strip()
    sex        = str(row["sex"]).lower().strip()
    state      = str(row["state"]).strip()
    age        = int(row["age"])try:
    results_df
except NameError:
    results_df = pd.read_csv("../results/week2_manual_tests.csv")

print("=== CORRECT predictions — reasoning traces ===")
for _, row in results_df[results_df["correct"] == True].head(2).iterrows():
    print(f"Input:     {row['input']}")
    print(f"Label:     {row['true_label']}")
    print(f"Reasoning: {row['trace']}")
    print()

print("\n=== WRONG predictions — reasoning traces ===")
for _, row in results_df[results_df["correct"] == False].head(2).iterrows():
    print(f"Input:     {row['input']}")
    print(f"True:      {row['true_label']} | Predicted: {row['pred_label']}")
    print(f"Reasoning: {row['trace']}")
    print()

    return (
        f"A {age}-year-old {sex}, {marital}, "
        f"working as a {occupation}. "
        f"Located in {state}."
    )
'''

prompts_code = '''
from src.serialize import serialize_row

SYSTEM_PROMPT = """You are an education level classifier. Given a person\'s demographic information,
predict whether they are college-educated (have a bachelor\'s degree or higher) or not.
Think step by step, then respond with ONLY one of these two labels on the final line: college or not_college."""


def build_zero_shot_prompt(row: dict) -> list:
    description = serialize_row(row)
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": (
            f"Classify this person\'s education level:\\n\\n"
            f"{description}\\n\\n"
            f"Answer with college or not_college only."
        )}
    ]


def build_few_shot_prompt(row: dict, examples: list) -> list:
    example_text = ""
    for ex in examples:
        example_text += f"Person: {serialize_row(ex[\'row\'])}\\nLabel: {ex[\'label\']}\\n\\n"
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": (
            f"Here are some labeled examples:\\n\\n{example_text}"
            f"Now classify this person:\\n\\n"
            f"{serialize_row(row)}\\n\\n"
            f"Answer with college or not_college only."
        )}
    ]
'''

parse_code = '''
import re

def parse_response(content: str) -> tuple:
    """
    Extract label and reasoning trace from Nemotron\'s response.
    Returns: (label, trace)
    """
    think_match = re.search(r"<think>(.*?)</think>", content, re.DOTALL)
    trace = think_match.group(1).strip() if think_match else ""
    label_match = re.search(r"\\b(not_college|college)\\b", content, re.IGNORECASE)
    label = label_match.group(1).lower() if label_match else "UNKNOWN"
    return label, trace
'''

src_dir = "../src"
with open(f"{src_dir}/serialize.py", "w") as f:
    f.write(serialize_code.strip())
with open(f"{src_dir}/prompts.py", "w") as f:
    f.write(prompts_code.strip())
with open(f"{src_dir}/parse_response.py", "w") as f:
    f.write(parse_code.strip())

print("Saved:")
print("  src/serialize.py")
print("  src/prompts.py")
print("  src/parse_response.py")

Saved:
  src/serialize.py
  src/prompts.py
  src/parse_response.py
